# Precise Lattice Parameters: Why Averaging Fails, and What To Do Instead

Ask a room of materials scientists how to get a lattice parameter from a diffractogram and most
will say: compute $a$ from each peak and average them. It is the obvious thing to do, it is what
the arithmetic invites, and on a real instrument it is wrong by about two orders of magnitude more
than strain measurement can tolerate.

This notebook shows why, on data whose true answer is known because we put it there. Then it builds
up the methods that work, one physical effect at a time, and finishes with a hexagonal cell where
the averaging question cannot even be *asked*.

The argument in one line: differentiating Bragg's law gives

$$\frac{\Delta d}{d} = -\cot\theta\,\Delta\theta$$

so a fixed angular error produces a **$\theta$-dependent** spacing error. The errors that dominate
a laboratory scan are systematic. Averaging a bias does not remove it.

The theory behind everything here is
{doc}`../../theory/precise_lattice_parameter_determination`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pytex import FrameDomain, ReferenceFrame
from pytex.core.fixtures import get_phase_fixture
from pytex.diffraction.xrd import RadiationSpec, generate_xrd_pattern
from pytex.diffraction.xrd_corrections import (
    profile_view,
    specimen_displacement_shift_deg,
    strip_kalpha2,
)
from pytex.diffraction.xrd_indexing import index_peaks
from pytex.diffraction.xrd_instrument import InstrumentBroadening
from pytex.diffraction.xrd_lattice_parameter import (
    determine_lattice_parameters,
    determine_lattice_parameters_from_pattern,
    determine_lattice_parameters_le_bail,
    extrapolation_values,
    nelson_riley_extrapolation,
)
from pytex.diffraction.xrd_measurement import MeasuredPowderPattern
from pytex.diffraction.xrd_peaks import detect_and_fit_peaks, detect_peaks

plt.rcParams["figure.dpi"] = 110
CRYSTAL = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
RADIUS_MM = 240.0

## 1. A scan whose answer we know

Everything below is checked against a number chosen in advance rather than against a previous run
of this code. We take nickel from the tracked phase fixtures, generate its Cu K$\alpha_1$/K$\alpha_2$
pattern, add a flat background and Poisson counting noise — and then displace the specimen by a
known 100 micrometres, which is a *perfectly ordinary* specimen preparation error.

A 100 µm displacement on a 240 mm goniometer moves the peaks by

$$\Delta(2\theta) = -\frac{2 s \cos\theta}{R}$$

which is tens of millidegrees at low angle, falling towards zero at back-reflection. Hold on to
that shape: it is the whole story.


In [ ]:
phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=CRYSTAL)
radiation = RadiationSpec.cu_ka_doublet()
a_true = phase.lattice.a
print(f"nickel from the fixture corpus: a = {a_true:.6f} A, {phase.symmetry.to_point_group().hermann_mauguin}")

ideal = generate_xrd_pattern(
    phase,
    radiation=radiation,
    two_theta_range_deg=(25.0, 150.0),
    resolution_deg=0.01,
    broadening_fwhm_deg=0.12,
    profile="pseudo_voigt",
    max_index=6,
)
axis = np.asarray(ideal.two_theta_grid_deg)
counts = np.random.default_rng(5).poisson(
    np.asarray(ideal.intensity_grid) / ideal.intensity_grid.max() * 30_000.0 + 150.0
).astype(float)

DISPLACEMENT_MM = 0.10
shift = specimen_displacement_shift_deg(
    axis, displacement_mm=DISPLACEMENT_MM, goniometer_radius_mm=RADIUS_MM
)
measured = MeasuredPowderPattern(
    name="nickel, specimen 100 um off axis",
    two_theta_deg=axis + shift,
    intensity=counts,
    radiation=radiation,
    synthetic=True,
)
print(f"the injected displacement moves peaks by {1000 * shift[0]:+.1f} mdeg at "
      f"{axis[0]:.0f} deg and {1000 * shift[-1]:+.1f} mdeg at {axis[-1]:.0f} deg")

## 2. Reading the pattern: the square-root ordinate is not decoration

Before any analysis, look at the data. The choice of vertical scale is a statistical decision, not
an aesthetic one. On a linear axis scaled to a 30 000-count peak, a 150-count background is a line
on the axis and a weak reflection is invisible. On a **square-root** axis — the variance-stabilising
transform for Poisson counts — the noise amplitude is the same everywhere, so a feature that *looks*
significant *is* significant.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6), constrained_layout=True)
for ax, scale in zip(axes, ("linear", "sqrt", "log10"), strict=True):
    view = profile_view(measured, scale=scale)
    ax.plot(view.abscissa, view.ordinate, lw=0.7, color="#2563eb")
    ax.set_title(scale)
    ax.set_xlabel(view.abscissa_label)
    ax.set_ylabel(view.ordinate_label.split(" (")[0])
fig.suptitle("The same data, three ordinates. Only one shows the weak reflections honestly.")

The abscissa is a choice too. $2\theta$ is what the instrument measured; $d$ and $Q$ are what the
crystal did. $Q = 4\pi\sin\theta/\lambda$ is wavelength-free, so a Cu and a Mo pattern of the same
phase superimpose on it. And $\sin^2\theta$ is the abscissa in which Bragg's law becomes *linear in
the cell parameters* — which is why the determination in section 6 works there.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.4), constrained_layout=True)
for ax, abscissa in zip(axes, ("d_angstrom", "q_inv_angstrom", "sin_squared_theta"), strict=True):
    view = profile_view(measured, abscissa=abscissa, scale="sqrt")
    ax.plot(view.abscissa, view.ordinate, lw=0.7, color="#7c3aed")
    ax.set_xlabel(view.abscissa_label)
    ax.set_ylabel("sqrt(counts)")
fig.suptitle("d runs backwards; Q is wavelength-free; sin^2(theta) is where the algebra is linear")

## 3. K$\alpha_2$: strip it to look, model it to fit

The doublet separates as $\tan\theta$, so above about 90° the K$\alpha_2$ line is a peak in its own
right. Rachinger stripping removes it by the recursion

$$I_1(2\theta) = I(2\theta) - r\,I_1(2\theta'), \qquad
\sin\theta' = \frac{\lambda_1}{\lambda_2}\sin\theta$$

and it works well enough to make a pattern much easier to read. Look at what it leaves behind,
though: a ringing train above the strongest reflection, because whatever the recursion fails to
remove at one angle is re-subtracted at the partner of *that* angle, and again beyond it.


In [ ]:
stripped = strip_kalpha2(measured)
window = (np.asarray(measured.two_theta_deg) > 142.0) & (np.asarray(measured.two_theta_deg) < 150.0)

fig, ax = plt.subplots(figsize=(9, 3.8), constrained_layout=True)
ax.plot(np.asarray(measured.two_theta_deg)[window], np.asarray(measured.intensity)[window],
        lw=1.0, color="#94a3b8", label="measured, alpha1 + alpha2")
ax.plot(np.asarray(stripped.two_theta_deg)[window], np.asarray(stripped.intensity)[window],
        lw=1.2, color="#dc2626", label="Rachinger stripped")
ax.set_xlabel("2*theta (deg)")
ax.set_ylabel("counts")
ax.set_title("The alpha2 line goes; a ringing residue stays")
ax.legend()

partner = float(np.rad2deg(2 * np.arcsin(
    (1.544390 / 1.540562) * np.sin(np.deg2rad(0.5 * 144.656)))))
index = int(np.argmin(np.abs(np.asarray(measured.two_theta_deg) - partner)))
before = float(np.asarray(measured.intensity)[index])
after = float(np.asarray(stripped.intensity)[index])
print(f"at the alpha2 position {partner:.2f} deg: {before:.0f} -> {after:.0f} counts "
      f"({100 * (1 - after / before):.0f}% removed)")

Stripping also *raises the noise*: each subtraction feeds the next, so
$\operatorname{var} I_1(2\theta) = \operatorname{var} I(2\theta) + r^2\operatorname{var} I_1(2\theta')$.
The library therefore treats stripping as a display convenience and **models** the doublet on every
fitting path. Modelling costs no extra parameter — Bragg's law at fixed $d$ fixes the partner
position — and it leaves the data untouched.

## 4. Finding the peaks, and knowing how well

Detection uses the instrument's own resolution function: the pattern is background-subtracted,
Anscombe-transformed so Poisson noise is homoscedastic at every count level, then convolved with
scale-matched Ricker kernels whose width is the Caglioti FWHM at that angle. Because the kernels
have unit $L^2$ norm, the detection threshold is quotable in **noise sigmas** rather than counts.

Each candidate is then fitted with a pseudo-Voigt, a local straight background, and the K$\alpha_2$
partner. What comes out is a position *and its standard uncertainty* — the number that makes the
whole determination weightable.


In [ ]:
instrument = InstrumentBroadening.ideal(0.12)
table = detect_and_fit_peaks(measured, instrument=instrument, prominence_sigma=5.0)
print(table.describe())
print()
for peak in table:
    print(f"  {peak.two_theta_deg:9.4f} +/- {peak.two_theta_standard_uncertainty_deg:.5f} deg"
          f"   FWHM {peak.fwhm_deg:.4f}   reduced chi2 {peak.reduced_chi_squared:5.2f}")

Detection also has to *refuse* the K$\alpha_2$ lines. Letting one in is not a spurious table row —
it is a reflection recorded at the wrong $d$ spacing, which is a lattice-parameter bias. Compare the
raw candidate list with the suppressed one:


In [ ]:
raw = detect_peaks(measured, instrument=instrument, suppress_kalpha2=False)
kept = detect_peaks(measured, instrument=instrument, suppress_kalpha2=True)
print(f"{len(raw)} candidates before alpha2 suppression, {len(kept)} after, "
      f"for {len(ideal.reflections)} real reflections")
print("suppressed:", [round(value, 2) for value in raw if
                      min(abs(value - other) for other in kept) > 0.05])

## 5. Indexing, and the residual that names the fault

Now assign Miller indices. The assignment is global rather than greedy — the Hungarian algorithm
minimises the total discrepancy over all one-to-one pairings, so it cannot give two peaks the same
reflection or strand a true partner.

Read the residual column before believing anything. Here every residual carries the same sign and
falls towards back-reflection, which is the signature of an instrument error rather than a wrong
cell — and `describe()` says so without being asked.


In [ ]:
indexing = index_peaks(table, phase, phase_name="nickel")
print(indexing.describe())
print()
for item in indexing:
    print(f"  ({''.join(str(v) for v in item.miller_indices)})  "
          f"obs {item.peak.two_theta_deg:8.4f}   calc {item.two_theta_calculated_deg:8.4f}   "
          f"residual {1000 * item.delta_two_theta_deg:+7.1f} mdeg   "
          f"({item.normalized_residual:+7.1f} sigma)")

## 6. The comparison this notebook exists for

Same peaks, same indices, five different assumptions about the systematic error. The true answer is
known: it is the fixture's own $a$.

The identity that makes this work is worth stating. If an aberration gives
$\Delta d/d = -K f(\theta)$, then because
$\Delta(\sin^2\theta)/\sin^2\theta = -2\,\Delta d/d$,

$$\sin^2\theta_\mathrm{obs}
= \frac{\lambda^2}{4}\,\mathbf{h}^{\mathsf{T}}\mathbf{G}^{*}\mathbf{h}
+ D\,\sin^2\theta\,f(\theta)$$

— so *whatever function the classical graphical method plots $a$ against is the same function that
appears here as a design column*. Every admissible $f$ vanishes at $\theta = 90^\circ$, which is
why the fitted cell is the extrapolated one.

For a specimen displacement the exact $f$ is $\cos^2\theta/\sin\theta$. Watch what happens when we
use it, and when we do not.


In [ ]:
rows = []
for method, extrapolation in (
    ("average", "none"),
    ("cohen", "none"),
    ("cohen", "bradley_jay"),
    ("cohen", "nelson_riley"),
    ("cohen", "cos_squared_over_sin"),
):
    result = determine_lattice_parameters(
        indexing, phase, method=method, extrapolation=extrapolation
    )
    rows.append((method, extrapolation, result))

print(f"{'method':9s} {'f(theta)':22s} {'a (A)':>10s} {'error':>11s} {'relative':>10s} "
      f"{'chi2':>9s}")
for method, extrapolation, result in rows:
    error = result.a - a_true
    print(f"{method:9s} {extrapolation:22s} {result.a:10.6f} {1e6 * error:+9.1f} uA "
          f"{abs(error) / a_true:10.1e} {result.reduced_chi_squared:9.2f}")

Three and a half orders of magnitude, from the same six peak positions. Nothing about the averaging
arithmetic is wrong; it simply has no mechanism for removing a $\theta$-dependent bias.

Note the reduced $\chi^2$ column. On real data you cannot check against a known answer — but you
*can* read the goodness of fit, and here it tracks the accuracy exactly. That is the number to
trust.

## 7. The picture: what the classical extrapolation plot actually shows

For a cubic cell a lattice parameter per reflection exists, so the classical construction can be
drawn: plot each one against $f(\theta)$ and extrapolate to zero. The **scatter** of the points
about the line is the random error; the **slope** is the systematic one. Averaging the points lands
on their mean. The answer is the intercept.


In [ ]:
plot = nelson_riley_extrapolation(indexing, phase, function="cos_squared_over_sin")
matched = determine_lattice_parameters(
    indexing, phase, extrapolation="cos_squared_over_sin"
)
naive = determine_lattice_parameters(indexing, phase, method="average")

fig, ax = plt.subplots(figsize=(8.5, 4.6), constrained_layout=True)
abscissa = np.asarray(plot["extrapolation_function"])
ordinate = np.asarray(plot["lattice_parameter"])
grid = np.linspace(0.0, abscissa.max() * 1.05, 50)
ax.plot(grid, plot["intercept"] + plot["slope"] * grid, color="#dc2626", lw=1.6,
        label="fitted line")
ax.scatter(abscissa, ordinate, s=42, color="#2563eb", zorder=3, label="per reflection")
ax.axhline(a_true, color="#15803d", ls="--", lw=1.2, label=f"true a = {a_true:.6f} A")
ax.axhline(naive.a, color="#f59e0b", ls=":", lw=1.4,
           label=f"average = {naive.a:.6f} A")
ax.scatter([0.0], [matched.a], s=110, facecolors="none", edgecolors="#dc2626", lw=1.8,
           zorder=4, label=f"extrapolated = {matched.a:.6f} A")
ax.set_xlabel(r"$\cos^2\theta / \sin\theta$")
ax.set_ylabel(r"$a$ from each reflection ($\mathrm{\AA}$)")
ax.set_title("The answer is where the line meets zero, not where the points average")
ax.legend(fontsize=8)

The green and red lines coincide; the orange one does not. That gap is the entire cost of averaging.

## 8. What the drift term actually removed

A refined coefficient is abstract. The angular shift it took out of each reflection is not — and
comparing that with the position uncertainties says immediately whether refining it was worth the
parameter.


In [ ]:
removed = matched.systematic_shift_deg
injected = specimen_displacement_shift_deg(
    matched.two_theta_deg, displacement_mm=DISPLACEMENT_MM, goniometer_radius_mm=RADIUS_MM
)
fig, ax = plt.subplots(figsize=(8.5, 3.8), constrained_layout=True)
ax.plot(matched.two_theta_deg, 1000 * injected, "o--", color="#15803d",
        label="injected aberration")
ax.plot(matched.two_theta_deg, 1000 * removed, "s-", color="#dc2626",
        label="removed by the drift term")
ax.errorbar(matched.two_theta_deg, np.zeros_like(matched.two_theta_deg),
            yerr=1000 * table.standard_uncertainty_deg, fmt="none", ecolor="#94a3b8",
            capsize=3, label="position uncertainties")
ax.set_xlabel("2*theta (deg)")
ax.set_ylabel("shift (mdeg)")
ax.set_title("The correction is a hundred times the uncertainty it competes with")
ax.legend(fontsize=8)

print(matched.describe())

The two curves differ by a constant, which is exactly right: a constant offset in $2\theta$ is
degenerate with a change in the cell, so the determination cannot distinguish "displaced specimen"
from "slightly different $a$" *except* through the shape. It absorbs the shape and leaves the
constant to the cell — and because the shape is what carries the $\theta$-dependence, that is
sufficient.

## 9. Hexagonal: where the question cannot even be asked

Outside the cubic system a lattice parameter *per reflection* does not exist, because one reflection
cannot determine both $a$ and $c$. The library refuses rather than returning a number.


In [ ]:
zirconium = get_phase_fixture("zr_hcp").load_phase(crystal_frame=CRYSTAL)
zr_pattern = generate_xrd_pattern(
    zirconium,
    radiation=radiation,
    two_theta_range_deg=(25.0, 150.0),
    resolution_deg=0.01,
    broadening_fwhm_deg=0.12,
    profile="pseudo_voigt",
    max_index=6,
)
zr_axis = np.asarray(zr_pattern.two_theta_grid_deg)
zr_counts = np.random.default_rng(9).poisson(
    np.asarray(zr_pattern.intensity_grid) / zr_pattern.intensity_grid.max() * 30_000.0 + 150.0
).astype(float)
zr_measured = MeasuredPowderPattern(
    name="zirconium, specimen 100 um off axis",
    two_theta_deg=zr_axis + specimen_displacement_shift_deg(
        zr_axis, displacement_mm=DISPLACEMENT_MM, goniometer_radius_mm=RADIUS_MM
    ),
    intensity=zr_counts,
    radiation=radiation,
    synthetic=True,
)

try:
    determine_lattice_parameters_from_pattern(
        zr_measured, zirconium, method="average", instrument=instrument
    )
except ValueError as error:
    print("refused, and correctly:")
    print(" ", error)

The joint solution is the only kind available. Because
$\mathbf{G}^{*}$ for a hexagonal cell has $a^{*} = b^{*}$ and $\gamma^{*} = 60^\circ$, so
$G^{*}_{12} = G^{*}_{11}/2$, the quadratic form collapses to the familiar
$A(h^2 + hk + k^2) + Cl^2$ — as a *consequence* of the symmetry, not as a special case anyone wrote
down.


In [ ]:
reciprocal = zirconium.lattice.reciprocal_metric_tensor()
print("G* for hexagonal zirconium:")
print(np.round(reciprocal, 8))
print(f"\nG*_12 / G*_11 = {reciprocal[0, 1] / reciprocal[0, 0]:.6f}  (exactly 1/2 by symmetry)")

for indices in [(1, 0, 0), (1, 1, 0), (1, 0, 2), (2, 1, 3)]:
    h, k, l_index = indices
    vector = np.array(indices, dtype=float)
    quadratic = float(vector @ reciprocal @ vector)
    textbook = reciprocal[0, 0] * (h * h + h * k + k * k) + reciprocal[2, 2] * l_index ** 2
    print(f"  {indices}: h^T G* h = {quadratic:.8f}, A(h^2+hk+k^2)+Cl^2 = {textbook:.8f}")

In [ ]:
zr_result, zr_indexing = determine_lattice_parameters_from_pattern(
    zr_measured,
    zirconium,
    method="cohen",
    extrapolation="cos_squared_over_sin",
    instrument=instrument,
    phase_name="zirconium",
)
print(f"indexed {zr_indexing.indexed_count} reflections")
print(f"a = {zr_result.a:.6f} +/- {zr_result.a_standard_uncertainty:.6f} A "
      f"(true {zirconium.lattice.a:.6f}, error {1e6 * (zr_result.a - zirconium.lattice.a):+.1f} uA)")
print(f"c = {zr_result.c:.6f} +/- {zr_result.c_standard_uncertainty:.6f} A "
      f"(true {zirconium.lattice.c:.6f}, error {1e6 * (zr_result.c - zirconium.lattice.c):+.1f} uA)")
print(f"c/a = {zr_result.axial_ratio:.6f} (true {zirconium.lattice.c / zirconium.lattice.a:.6f})")

## 10. Le Bail: when the peaks stop being separable

A hexagonal pattern overlaps badly enough that single-peak fitting runs out of *resolvable* lines
long before the reflection list runs out of reflections. Whole-pattern decomposition uses every
measured point and **extracts** the reflection intensities rather than modelling them, so neither
texture nor a wrong atomic basis can bias the cell.

Its diagnostic is the difference curve, not a residual per reflection — because it never measures an
individual peak position at all.


In [ ]:
le_bail = determine_lattice_parameters_le_bail(
    zr_measured,
    zirconium,
    systematic="displacement",
    goniometer_radius_mm=RADIUS_MM,
    cycles=10,
)
print(f"refined displacement = {le_bail.drift_coefficient:.5f} mm  "
      f"(injected {DISPLACEMENT_MM:.5f} mm)")
print(f"a = {le_bail.a:.6f} A, error {1e6 * (le_bail.a - zirconium.lattice.a):+.1f} uA")
print(f"c = {le_bail.c:.6f} A, error {1e6 * (le_bail.c - zirconium.lattice.c):+.1f} uA")
print(f"reduced chi2 = {le_bail.reduced_chi_squared:.3f}, "
      f"R_wp = {le_bail.weighted_profile_r:.4f} on the subtracted profile")

The fit returns the aberration **in millimetres**, matching what we injected. That is a far stronger
check than "the lattice parameter looks right": a wrong model can land on a right-looking cell by
compensating errors, but it will not also reproduce a physical quantity nobody fitted for directly.

And the goodness of fit must fail loudly when the model is wrong. Leave the displacement out:


In [ ]:
ignored = determine_lattice_parameters_le_bail(
    zr_measured, zirconium, systematic="none", cycles=10
)
print(f"{'systematic term':22s} {'a error':>12s} {'c error':>12s} {'chi2':>9s}")
for label, result in (("displacement refined", le_bail), ("none", ignored)):
    print(f"{label:22s} {1e6 * (result.a - zirconium.lattice.a):+10.1f} uA "
          f"{1e6 * (result.c - zirconium.lattice.c):+10.1f} uA {result.reduced_chi_squared:9.3f}")

In [ ]:
fig, (top, bottom) = plt.subplots(
    2, 1, figsize=(11, 5.4), sharex=True, height_ratios=(3, 1), constrained_layout=True
)
angles = np.asarray(le_bail.profile_two_theta_deg)
observed = np.asarray(le_bail.profile_observed)
calculated = np.asarray(le_bail.profile_calculated)
top.plot(angles, observed, lw=0.7, color="#94a3b8", label="observed - background")
top.plot(angles, calculated, lw=1.0, color="#2563eb", label="Le Bail calculated")
top.set_ylabel("counts")
top.legend(fontsize=8)
top.set_title("Whole-pattern decomposition of hexagonal zirconium")
bottom.plot(angles, observed - calculated, lw=0.7, color="#7c3aed")
bottom.axhline(0.0, color="#94a3b8", lw=0.8)
bottom.set_xlabel("2*theta (deg)")
bottom.set_ylabel("difference")

## 11. A lattice parameter is not a stress

The reason most people want a precise lattice parameter is stress, so it is worth being blunt about
what this notebook has and has not produced.

A symmetric $\theta$–$2\theta$ scan measures the spacing of planes whose normal is parallel to the
scattering vector — that is, **normal to the specimen surface**. One such measurement gives one
strain component. Converting it to a stress needs $d(hkl)$ measured at several specimen tilts
$\psi$, the slope of $d$ against $\sin^2\psi$, and the X-ray elastic constants
$\tfrac{1}{2}S_2$ and $S_1$ of the particular reflection used — which are $hkl$-dependent, because
the diffracting grains are an orientation-selected subset and are elastically anisotropic.

What this module supplies is the precise spacings that such an analysis consumes. Every result says
so itself:


In [ ]:
strain = matched.strain_relative_to_reference
print(f"lattice strain along a, against the reference cell: {strain:+.3e}")
print()
print(matched.describe())

## What to take away

1. **Averaging over reflections cannot remove a systematic error**, because
   $\Delta d/d = -\cot\theta\,\Delta\theta$ makes the error depend on angle. It divides the random
   scatter by $\sqrt{N}$ and leaves the bias untouched.
2. **Choose the extrapolation function to match the aberration.** $\cot\theta$ for a detector zero,
   $\cos^2\theta/\sin\theta$ for a specimen displacement, Nelson–Riley for a mixture. All vanish at
   $\theta = 90^\circ$, which is what makes the fit the extrapolation.
3. **One drift term removes one aberration.** A pattern carrying both a zero error and a
   displacement cannot be corrected by any single $f$ — which is why real practice calibrates the
   zero against a standard first.
4. **Read the reduced $\chi^2$.** On real data it is the only thing standing in for the known
   answer, and here it tracked the accuracy across four orders of magnitude.
5. **Model K$\alpha_2$, do not strip it**, on any path that fits.
6. **Use Le Bail when the peaks overlap**, and check that it returns the aberration in physical
   units, not merely a plausible cell.
7. **Strain is not stress.** The $\sin^2\psi$ analysis is a separate measurement.

The same methods are available in the workbench under **XRD → Determine lattice parameters**, where
the injected displacement is a slider and the comparison in section 6 is one click.
